# Batch energy data pipeline

## Stage 1 · Acquire public data


The electricity sector emissions and generation data for the last ten years (2014 to 2024) from the National Greenhouse and Energy Reporting (NGER) website

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path("..") if Path.cwd().name == "batch_pipeline" else Path(".")
BATCH_DIR = PROJECT_ROOT / "data" / "batch"
BASE_DIR = BATCH_DIR / "Task1_output"
BASE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
NGER_DATASETS = {
    "2023-24": "ID0243",
    "2022-23": "ID0083",
    "2021-22": "ID0082",
    "2020-21": "ID0081",
    "2019-20": "ID0080",
    "2018-19": "ID0079",
    "2017-18": "ID0078",
    "2016-17": "ID0077",
    "2015-16": "ID0076",
    "2014-15": "ID0075",
}


In [ ]:
def download_nger_10_years(save_dir):
    all_years = []

    for year, dataset_id in NGER_DATASETS.items():
        print(f"Downloading NGER {year}...")

        url = (
            "https://api.cer.gov.au/datahub-public/v1/api/"
            f"ODataDataset/NGER/dataset/{dataset_id}?select%3D%2A"
        )

        response = requests.get(url)
        response.raise_for_status()

        data = response.json()
        df = pd.DataFrame(data)
        df["reporting_year"] = year

        # Save single-year CSV
        yearly_path = os.path.join(save_dir, f"NGER_{year}.csv")
        df.to_csv(yearly_path, index=False)

        all_years.append(df)

    # Merge ten years
    combined_df = pd.concat(all_years, ignore_index=True)

    combined_path = os.path.join(save_dir, "NGER_2014_2024_combined.csv")
    combined_df.to_csv(combined_path, index=False)

    return combined_df


In [ ]:
nger_df = download_nger_10_years(BASE_DIR)

In [ ]:
import pandas as pd
#load the combined dataset
NGER_2014_2024_combined_df=pd.read_csv(os.path.join(BASE_DIR, "NGER_2014_2024_combined.csv"))

print("Total rows:", NGER_2014_2024_combined_df.shape[0])
print(NGER_2014_2024_combined_df["reporting_year"].value_counts())
print(NGER_2014_2024_combined_df.head())

Data about current large-scale renewable power stations/projects by status from the Clean Energy Regulator (CER).


Current CER power stations/projects tables: approved, committed, probable.


In [ ]:
CER_STATUS_URLS = {
    "approved":  "https://cer.gov.au/document/power-stations-and-projects-accredited",
    "committed": "https://cer.gov.au/document/power-stations-and-projects-committed",
    "probable":  "https://cer.gov.au/document/power-stations-and-projects-probable",
}

cer_parts = []
for status, url in CER_STATUS_URLS.items():
    df_part = pd.read_csv(url)
    df_part["project_status"] = status
    cer_parts.append(df_part)
    print(f"Loaded {status}: shape={df_part.shape}")

cer_accredited_df = pd.concat(cer_parts, ignore_index=True)
print("\nCombined current CER dataset:", cer_accredited_df.shape)
print(cer_accredited_df.head())


In [ ]:
import requests
import os
import pandas as pd

os.makedirs(BASE_DIR, exist_ok=True)

cer_local_path = os.path.join(
    BASE_DIR,
    "CER_current_power_stations_status_combined.csv"
)

status_local_paths = {}
for status, url in CER_STATUS_URLS.items():
    out_file = os.path.join(BASE_DIR, f"CER_current_{status}_power_stations.csv")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(out_file, "wb") as f:
        f.write(response.content)
    status_local_paths[status] = out_file

cer_accredited_df.to_csv(cer_local_path, index=False)

print("Saved CER status files:")
for k, v in status_local_paths.items():
    print(f"  {k}: {v}")
print("  combined:", cer_local_path)


Data from the Australian Bureau of Statistics (ABS) Data by Regions portal

In [ ]:
ABS_POP_URL = (
    "https://www.abs.gov.au/methodologies/data-region-methodology/"
    "2011-24/14100DO0001_2011-24.xlsx"
)

In [ ]:
import requests
import os
import pandas as pd

os.makedirs(BASE_DIR, exist_ok=True)

abs_local_path = os.path.join(
    BASE_DIR,
    "ABS_population_people_2011_2024.xlsx"
)

response = requests.get(ABS_POP_URL)
response.raise_for_status()

with open(abs_local_path, "wb") as f:
    f.write(response.content)


In [ ]:
xls = pd.ExcelFile(abs_local_path)
print(xls.sheet_names)

In [ ]:
preview = pd.read_excel(
    abs_local_path,
    sheet_name="Table 1",
    nrows=15,
    header=None
)

print(preview)

In [ ]:
abs_pop_df = pd.read_excel(
    abs_local_path,
    sheet_name="Table 1",
    skiprows=6
)

In [ ]:
abs_pop_df.to_csv(
    os.path.join(BASE_DIR, "ABS_population_people_table1.csv"),
    index=False
)

print(abs_pop_df.shape)

### Supplementary ABS ERP Data (via ABS SDMX API)

In this part, the pipeline retrieves extra ERP data from the ABS SDMX API.
The ABS Data by Regions file has many missing ERP values before 2019.
The pipeline uses quarterly ERP and keep only Q2 (June quarter).
This match the "year ended 30 June" idea in ABS tables.

I do this because our cleaned Stage 2 table is aligned to 2014-2023.
Without this step, some years will have blanks and the join is weak.


In [ ]:
import requests as _req

# --- Fetch ERP from ABS SDMX API ---
# Dimensions: MEASURE=1 (ERP), SEX=3 (Persons), AGE=TOT (All ages),
# REGION=1..8 (NSW,VIC,QLD,SA,WA,TAS,NT,ACT), FREQ=Q (Quarterly)
_ABS_ERP_URL = (
    'https://data.api.abs.gov.au/rest/data/ABS,ERP_Q,1.0.0/'
    '1.3.TOT.1+2+3+4+5+6+7+8.Q'
    '?startPeriod=2014-Q2&endPeriod=2024-Q2&detail=full'
)
_headers = {'Accept': 'application/vnd.sdmx.data+json;version=2.0.0'}

print('Fetching ERP data from ABS SDMX API...')
_resp = _req.get(_ABS_ERP_URL, headers=_headers, timeout=30)
_resp.raise_for_status()
_data = _resp.json()

# Parse the SDMX-JSON response structure
_structure = _data['data']['structures'][0]
_dims = {d['id']: {i: v for i, v in enumerate(d['values'])}
         for d in _structure['dimensions']['series']}
_obs_times = {i: v for i, v in enumerate(
    _structure['dimensions']['observation'][0]['values'])}

# Map ABS numeric region codes to standard state abbreviations
_REGION_MAP = {
    '1': 'NSW', '2': 'VIC', '3': 'QLD', '4': 'SA',
    '5': 'WA',  '6': 'TAS', '7': 'NT',  '8': 'ACT',
}

# Extract ERP values, keeping only the June quarter (Q2) per year
_rows = []
_series = _data['data']['dataSets'][0]['series']
for _key, _val in _series.items():
    _idx = [int(x) for x in _key.split(':')]
    _region = _dims['REGION'][_idx[3]]
    _state = _REGION_MAP.get(_region['id'])
    for _ok, _ov in _val['observations'].items():
        _tp = _obs_times[int(_ok)]['id']  # e.g. '2014-Q2'
        if _tp.endswith('-Q2'):            # June quarter only
            _rows.append({'state': _state, 'year': int(_tp[:4]),
                          'estimated_resident_population_no': _ov[0]})

erp_supplement = pd.DataFrame(_rows)

# Save the supplementary ERP data
erp_supplement.to_csv(
    os.path.join(BASE_DIR, 'ABS_ERP_Q_supplement_2014_2024.csv'), index=False
)

print(f'Fetched {len(erp_supplement)} state-year ERP records')
print(erp_supplement.sort_values(['year', 'state']).head(10))


# Stage 2 · Clean, integrate, and validate matches

This workflow standardises source fields, matches eligible facilities, and joins
state-year population data. It keeps candidate matches and review diagnostics
so that coverage and uncertain cases can be inspected.


## Design Principles and Why They Matter

The workflow follows four design rules.

1. **Traceable steps**: keep intermediate tables and checks.
2. **Safer matching first**: avoid aggressive fuzzy matches.
3. **Validate before trusting**: look at coverage and flags.
4. **Keep the final export focused**: remove helper columns in final export.




In [ ]:
# Task2 setup: import libraries and set paths.
# We create the Task2 output folder first.
from pathlib import Path
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

BASE_DIR = BATCH_DIR
TASK1_DIR = BASE_DIR / "Task1_output"
TASK2_DIR = BASE_DIR / "Task2_output"
TASK2_DIR.mkdir(exist_ok=True)

print("BASE_DIR:", BASE_DIR)

In [ ]:
#functions used in cleaning and matching.

def coalesce(df, cols):
    out = pd.Series([pd.NA] * len(df), index=df.index, dtype="object")
    for c in cols:
        if c in df.columns:
            out = out.where(out.notna(), df[c])
    return out


STATE_CODES = {"NSW","VIC","QLD","SA","WA","TAS","NT","ACT"}


def normalize_state(s):
    if pd.isna(s):
        return pd.NA
    t = str(s).strip().upper()
    return t if t in STATE_CODES else pd.NA


def _base_clean(text):
    if pd.isna(text):
        return ""
    s = str(text).lower()
    s = s.replace("cmm", "waste coal mine gas")
    s = re.sub(r"\blfg\b", "landfill gas", s)
    s = re.sub(r"\bmt\b", "mount", s)
    s = re.sub(r"\bst\b", "saint", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_name_strict(text):
    s = _base_clean(text)
    s = re.sub(r"\b(power station|wind farm|solar farm|wind park|power plant|facility|project|pty|ltd|limited|australia)\b", " ", s)
    s = re.sub(r"\b(nsw|vic|qld|sa|wa|tas|nt|act)\b", " ", s)
    s = re.sub(r"\b\d{4}\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_name_relaxed(text):
    s = normalize_name_strict(text)
    s = re.sub(r"\b(stage|unit|block|lot|site|north|south|east|west)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


GENERIC_TOKENS = {
    "solar", "wind", "hydro", "energy", "hub", "farm", "station", "power", "project", "plant",
    "renewable", "park", "generation", "electric", "electricity", "unit", "stage", "site", "block",
    "creek", "bay", "point", "river", "valley", "north", "south", "east", "west", "pty", "ltd", "limited", "group", "holdings"
}


def informative_tokens(name_norm):
    if not isinstance(name_norm, str) or not name_norm:
        return set()
    toks = [t for t in name_norm.split() if len(t) >= 3 and t not in GENERIC_TOKENS]
    return set(toks)


def lexical_overlap_count(a_norm, b_norm):
    return len(informative_tokens(a_norm) & informative_tokens(b_norm))


def lexical_guard(a_norm, b_norm, mode="strict"):
    overlap = lexical_overlap_count(a_norm, b_norm)
    contains = (a_norm in b_norm or b_norm in a_norm) and min(len(a_norm), len(b_norm)) >= 6
    if mode == "rescue":
        return overlap >= 1 or contains
    return overlap >= 1 or contains


def split_name_parts(text):
    s = _base_clean(text)
    parts = re.split(r"\s*(?:&|and|/)\s*", s)
    parts = [normalize_name_strict(p) for p in parts]
    parts = [p for p in parts if p]
    return sorted(set(parts))


def extract_numbers(text):
    s = _base_clean(text)
    return tuple(sorted(set(re.findall(r"\d+", s))))


def numeric_compatible(cer_name, nger_name):
    a = extract_numbers(cer_name)
    b = extract_numbers(nger_name)
    if a and b and a != b:
        return False
    return True


def fuel_category_from_text(text):
    if pd.isna(text):
        return None
    s = _base_clean(text)
    if "solar" in s:
        return "Solar"
    if "wind" in s:
        return "Wind"
    if any(k in s for k in ["hydro", "water"]):
        return "Hydro"
    if any(k in s for k in ["biomass", "biogas", "bagasse", "wood", "landfill gas", "waste coal mine gas", "biofuel", "sludge"]):
        return "Biomass"
    return None


NGER_FUEL_MAP = {
    "solar": "Solar",
    "wind": "Wind",
    "hydro": "Hydro",
    "hydroelectricity": "Hydro",
    "biogas": "Biomass",
    "bagasse": "Biomass",
    "wood": "Biomass",
    "landfill gas": "Biomass",
    "waste coal mine gas": "Biomass",
    "sludge biogas": "Biomass",
    "biofuel": "Biomass",
}


def map_nger_fuel(primary_fuel):
    if pd.isna(primary_fuel):
        return None
    s = _base_clean(primary_fuel)
    if any(k in s for k in ["coal", "gas", "diesel", "kerosene", "battery", "liquid fuel"]):
        if s in ["landfill gas", "waste coal mine gas", "biogas", "sludge biogas"]:
            pass
        else:
            return None
    for k, v in NGER_FUEL_MAP.items():
        if k in s:
            return v
    return fuel_category_from_text(s)


def clean_columns(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
        .str.strip('_')
    )
    return out


def merge_duplicate_columns(df):
    """Coalesce columns that share the same name after cleaning."""
    if not df.columns.duplicated().any():
        return df
    seen = {}
    for i, col in enumerate(df.columns):
        seen.setdefault(col, []).append(i)
    result = pd.DataFrame(index=df.index)
    for col, idxs in seen.items():
        merged = df.iloc[:, idxs[0]]
        for j in idxs[1:]:
            merged = merged.combine_first(df.iloc[:, j])
        result[col] = merged
    return result


def replace_missing_tokens(df):
    return df.replace(['', ' ', '-', '--', 'NA', 'N/A', 'na', 'n/a', 'null', 'NULL'], pd.NA)


def standardise_state(series):
    mapping = {
        'new south wales': 'NSW',  'nsw': 'NSW',
        'victoria': 'VIC',         'vic': 'VIC',
        'queensland': 'QLD',       'qld': 'QLD',
        'south australia': 'SA',   'sa':  'SA',
        'western australia': 'WA', 'wa':  'WA',
        'tasmania': 'TAS',         'tas': 'TAS',
        'northern territory': 'NT','nt':  'NT',
        'australian capital territory': 'ACT', 'act': 'ACT',
    }
    return series.astype('string').str.strip().str.lower().map(mapping)


def parse_start_year(value):
    if pd.isna(value):
        return pd.NA
    m = re.search(r'(20\d{2}|19\d{2})', str(value))
    return int(m.group(1)) if m else pd.NA


ALIAS_MAP = {
    'reportingentity':            'entity_name',
    'controllingcorporation':     'entity_name',
    'facilityname':               'facility_name',
    'electricityproductiongj':    'prod_gj',
    'electricityproductionmwh':   'prod_mwh',
    'scope1tco2e':                'emissions_s1',
    'totalscope1emissionstco2e':  'emissions_s1',
    'scope2tco2e':                'emissions_s2',
    'totalscope2emissionstco2e':  'emissions_s2',
    'totalscope2emissionstco2e2': 'emissions_s2',
    'totalemissionstco2e':        'emissions_total',
    'emissionintensitytmwh':      'emission_intensity',
    'emissionintensitytco2emwh':  'emission_intensity',
    'gridconnected':              'grid_connected',
    'gridconnected2':             'grid_connected',
    'primaryfuel':                'primary_fuel',
    'importantnotes':             'important_notes',
}


def coalesce_aliases(df, alias_map):
    """Rename/merge columns using source->target alias map."""
    out = df.copy()
    for source, target in alias_map.items():
        if source not in out.columns or source == target:
            continue
        if target in out.columns:
            out[target] = out[target].combine_first(out[source])
            out = out.drop(columns=[source])
        else:
            out = out.rename(columns={source: target})
    return out


In [ ]:
#Load source files created in Task1.
nger_raw = pd.read_csv(TASK1_DIR / "NGER_2014_2024_combined.csv")
cer_raw = pd.read_csv(TASK1_DIR / "CER_current_power_stations_status_combined.csv")
abs_people_raw = pd.read_csv(TASK1_DIR / "ABS_population_people_table1.csv", low_memory=False)
abs_erp_raw = pd.read_csv(TASK1_DIR / "ABS_ERP_Q_supplement_2014_2024.csv")

print("NGER:", nger_raw.shape)
print("CER:", cer_raw.shape)
print("ABS people:", abs_people_raw.shape)
print("ABS ERP:", abs_erp_raw.shape)

## 1) Clean NGER

Steps:
- standardize changing column names across NGER years.
- parse `reporting_year` as the **start year** (example: `2014-15` -> `2014`).
- convert numeric fields and fix missing-value tokens.
- backfill `emissions_total` and `emission_intensity` when possible.
- add an outlier flag for very high emission intensity.

Why do this:
- Raw NGER schema is inconsistent between years.
- If year parsing is wrong by +1 year, later joins will be wrong too.
- This cleaner table is easier for matching and aggregation.


In [ ]:
# Clean NGER data: columns, year, state, and numeric fields.
# Keep fields needed for matching later.
nger = clean_columns(nger_raw)
nger = merge_duplicate_columns(nger)
nger = replace_missing_tokens(nger)
nger = coalesce_aliases(nger, ALIAS_MAP)

nger_clean = nger.copy()
nger_clean['year'] = nger_clean['reporting_year'].map(parse_start_year)
nger_clean['state'] = standardise_state(nger_clean['state'])
nger_clean['type'] = nger_clean['type'].astype('string').str.strip().str.upper()

for col in ['prod_gj', 'prod_mwh', 'emissions_s1', 'emissions_s2', 'emissions_total', 'emission_intensity']:
    if col in nger_clean.columns:
        nger_clean[col] = pd.to_numeric(
            nger_clean[col].astype('string').str.replace(',', '', regex=False),
            errors='coerce'
        )

has_scope_data = nger_clean['emissions_s1'].notna() | nger_clean['emissions_s2'].notna()
computed_total = nger_clean['emissions_s1'].fillna(0) + nger_clean['emissions_s2'].fillna(0)
nger_clean['emissions_total'] = nger_clean['emissions_total'].fillna(computed_total.where(has_scope_data))

nger_clean['emission_intensity'] = nger_clean['emission_intensity'].fillna(
    nger_clean['emissions_total'] / nger_clean['prod_mwh'].replace({0.0: np.nan})
)

ei_p99 = nger_clean['emission_intensity'].quantile(0.99)
nger_clean['emission_intensity_flagged'] = nger_clean['emission_intensity'] > ei_p99

nger_clean = nger_clean.dropna(subset=['year', 'state', 'facility_name']).drop_duplicates()

nger_clean['fuel_category'] = nger_clean['primary_fuel'].map(map_nger_fuel)
nger_clean['facility_name_norm_strict'] = nger_clean['facility_name'].map(normalize_name_strict)
nger_clean['facility_name_norm_relaxed'] = nger_clean['facility_name'].map(normalize_name_relaxed)
nger_clean['name_parts'] = nger_clean['facility_name'].map(split_name_parts)
nger_clean['name_numbers'] = nger_clean['facility_name'].map(extract_numbers)

nger = nger_clean.copy()

print(nger.shape)
print('NGER type distribution:')
print(nger['type'].value_counts(dropna=False))
print('NGER renewable fuel categories:')
print(nger['fuel_category'].value_counts(dropna=False))
print(f"NGER high-intensity outlier flags: {int(nger['emission_intensity_flagged'].sum())}")


## 2) Clean CER

Steps:
- combine CER fields from different source formats.
- normalize station names and state values.
- parse date columns into one consistent timeline.
- create `is_matchable_to_nger` with a policy-based rule.

Policy use now:
- still filter SGU-like rows because names are noisy.
- allow `approved` and `probable` to enter matching even if capacity is small.
- For other status, keep the small-capacity guard.



In [ ]:
#Clean CER data and build one consistent table.
#create normalized name fields for matching.
cer = pd.DataFrame()
cer["cer_unique_key"] = cer_raw.get("Accreditation code")
cer["facility_name"] = coalesce(cer_raw, ["Power station name", "Project Name"])
cer["state"] = coalesce(cer_raw, ["State", "State "]).map(normalize_state)
cer["postcode"] = pd.to_numeric(cer_raw.get("Postcode"), errors="coerce").astype("Int64")
cer["installed_capacity_mw"] = pd.to_numeric(coalesce(cer_raw, ["Installed capacity (MW)", "MW Capacity"]), errors="coerce")
cer["fuel_source"] = coalesce(cer_raw, ["Fuel Source(s)", "Fuel Source"])
cer["project_status"] = cer_raw.get("project_status").astype("string").str.strip().str.lower()
cer["accreditation_start_date"] = pd.to_datetime(cer_raw.get("Accreditation start date"), dayfirst=True, errors="coerce")
cer["approval_date"] = pd.to_datetime(cer_raw.get("Approval date"), dayfirst=True, errors="coerce")
cer["committed_date"] = pd.to_datetime(cer_raw.get("Committed Date (Month/Year)"), format="%b-%Y", errors="coerce")
cer["reference_date"] = cer["accreditation_start_date"].combine_first(cer["approval_date"]).combine_first(cer["committed_date"])
cer["accreditation_year"] = cer["reference_date"].dt.year.astype("Int64")

cer["fuel_category"] = cer["fuel_source"].map(fuel_category_from_text)
cer["facility_name_norm_strict"] = cer["facility_name"].map(normalize_name_strict)
cer["facility_name_norm_relaxed"] = cer["facility_name"].map(normalize_name_relaxed)
cer["name_parts"] = cer["facility_name"].map(split_name_parts)
cer["name_numbers"] = cer["facility_name"].map(extract_numbers)

print(cer.shape)
print("CER status distribution:")
print(cer["project_status"].value_counts(dropna=False))
print("CER fuel categories:")
print(cer["fuel_category"].value_counts(dropna=False))

cer_name_lc = cer["facility_name"].fillna("").astype(str).str.lower()
cer["is_sgu_like"] = cer_name_lc.str.contains(r"sgu", regex=True)
cer["is_small_capacity"] = cer["installed_capacity_mw"].notna() & (cer["installed_capacity_mw"] < 5)
cer["is_future_status"] = cer["project_status"].isin(["approved", "probable"])
cer["is_matchable_to_nger"] = (~cer["is_sgu_like"]) & (cer["is_future_status"] | ~cer["is_small_capacity"])
print("CER matchable flag counts:")
print(cer["is_matchable_to_nger"].value_counts(dropna=False))
print("CER matchable by status:")
print(pd.crosstab(cer["project_status"], cer["is_matchable_to_nger"], dropna=False))


## 3) Clean ABS (Keep All Columns)

Steps:
- I keep almost all ABS people columns for context.
- I clean `state` and `year` keys.
- I keep one best row when a state-year appears more than one time.
- I build a canonical target grid from NGER state-year keys.
- I left-join ABS and ERP supplement onto this grid.

Why I do this:
- It guarantees one row per state-year in the analysis window.
- It avoids row inflation from outer joins.
- ERP backfill makes population coverage complete for the window.

This part is a bit long, but it help my final join become stable.


In [ ]:
# Clean ABS data at state-year level.
# Fill missing population values with ERP where possible.
abs_df = clean_columns(abs_people_raw)
abs_df = replace_missing_tokens(abs_df)

state_labels = {
    'new south wales': 'NSW',
    'victoria': 'VIC',
    'queensland': 'QLD',
    'south australia': 'SA',
    'western australia': 'WA',
    'tasmania': 'TAS',
    'northern territory': 'NT',
    'australian capital territory': 'ACT',
}

abs_df['label_clean'] = abs_df['label'].astype('string').str.strip().str.lower()
abs_state = abs_df[abs_df['label_clean'].isin(state_labels.keys())].copy()
abs_state['state'] = abs_state['label_clean'].map(state_labels)
abs_state['year'] = pd.to_numeric(abs_state['year'], errors='coerce').astype('Int64')

abs_state_year = abs_state.drop(columns=['code', 'label', 'label_clean'], errors='ignore').copy()
for col in abs_state_year.columns:
    if col not in ['state', 'year']:
        abs_state_year[col] = pd.to_numeric(abs_state_year[col], errors='coerce')

abs_state_year = abs_state_year.dropna(subset=['state', 'year']).drop_duplicates()

abs_state_year = (
    abs_state_year
    .assign(_n_valid=lambda df: df.notna().sum(axis=1))
    .sort_values(['state', 'year', '_n_valid'], ascending=[True, True, False])
    .drop_duplicates(subset=['state', 'year'], keep='first')
    .drop(columns=['_n_valid'])
)

nger_state_year = (
    nger[['state', 'year']]
    .dropna()
    .drop_duplicates()
    .sort_values(['state', 'year'])
)

target_states = sorted(nger_state_year['state'].dropna().unique())
target_years = sorted(int(y) for y in nger_state_year['year'].dropna().unique())
target_grid = pd.DataFrame([(s, y) for s in target_states for y in target_years], columns=['state', 'year'])

abs_on_grid = target_grid.merge(abs_state_year, on=['state', 'year'], how='left')

abs_erp = abs_erp_raw.copy()
abs_erp['state'] = standardise_state(abs_erp['state'])
abs_erp['year'] = pd.to_numeric(abs_erp['year'], errors='coerce').astype('Int64')
abs_erp['estimated_resident_population_no'] = pd.to_numeric(abs_erp['estimated_resident_population_no'], errors='coerce')
abs_erp = abs_erp.dropna(subset=['state', 'year']).drop_duplicates(subset=['state', 'year'])

abs_on_grid = abs_on_grid.merge(
    abs_erp[['state', 'year', 'estimated_resident_population_no']].rename(
        columns={'estimated_resident_population_no': 'estimated_resident_population_no_erp'}
    ),
    on=['state', 'year'],
    how='left',
)

if 'estimated_resident_population_no' in abs_on_grid.columns:
    abs_on_grid['estimated_resident_population_no'] = abs_on_grid['estimated_resident_population_no'].combine_first(
        abs_on_grid['estimated_resident_population_no_erp']
    )

abs_on_grid['abs_estimated_resident_population_no'] = abs_on_grid['estimated_resident_population_no']

abs_people_cols = [c for c in abs_on_grid.columns if c not in ['state', 'year', 'estimated_resident_population_no_erp', 'abs_estimated_resident_population_no']]
rename_map = {c: f'abs_people_{c}' for c in abs_people_cols}
abs_clean = abs_on_grid.rename(columns=rename_map)

if 'abs_people_estimated_resident_population_no' in abs_clean.columns:
    abs_clean['estimated_resident_population_no'] = abs_clean['abs_people_estimated_resident_population_no']

abs_clean = abs_clean.sort_values(['state', 'year']).reset_index(drop=True)
print(abs_clean.shape)
print(f"ABS ERP null count after supplement: {int(abs_clean['estimated_resident_population_no'].isna().sum())}")


## 4) Build Unique Facility Lists

What this section does:
- Converts row-level data into unique `(state, facility_name)` sets before fuzzy matching.

Why:
- This greatly reduces compute cost and avoids repeating the same fuzzy operation for each year row.


In [ ]:
# Build unique facility lists before fuzzy matching.
# This is faster and easier to join back later.
nger_fac = (
    nger[["state", "facility_name", "facility_name_norm_strict", "facility_name_norm_relaxed", "name_parts", "name_numbers", "fuel_category"]]
    .dropna(subset=["state", "facility_name"])
    .drop_duplicates(["state", "facility_name"])
    .reset_index(drop=True)
)

cer_fac = (
    cer[["state", "facility_name", "facility_name_norm_strict", "facility_name_norm_relaxed", "name_parts", "name_numbers", "fuel_category", "is_matchable_to_nger", "project_status", "accreditation_year"]]
    .dropna(subset=["state", "facility_name"])
    .drop_duplicates(["state", "facility_name"])
    .reset_index(drop=True)
)

print("Unique NGER facilities:", len(nger_fac))
print("Unique CER facilities:", len(cer_fac))


## 5) Load Manual Overrides

What do in this section:
- load manually checked CER-to-NGER mapping files.
- clean key columns and remove duplicated pairs.
- apply these overrides first in the match flow.

Why do this:
- Human-checked pairs are usually more reliable.
- It improves recall without adding too much noise.



In [ ]:
# Load manual override mappings.
# Manual approved pairs have higher priority than fuzzy rules.
EMBEDDED_APPROVED_OVERRIDES = [
    {"state": "QLD", "cer_facility_name": "Clarke Creek Wind Farm Stage 2", "nger_facility_name": "Clarke Creek Wind Farm"},
    {"state": "SA",  "cer_facility_name": "Willogoleche 2 Wind Farm",      "nger_facility_name": "Willogoleche Wind Farm"},
    {"state": "TAS", "cer_facility_name": "Northern Midlands Solar Farm",  "nger_facility_name": "Midlands Power Station"},
    {"state": "VIC", "cer_facility_name": "Kiamal Solar Farm - Stage 2",   "nger_facility_name": "Kiamal Solar Farm"},
    {"state": "VIC", "cer_facility_name": "Winton North Solar Farm",       "nger_facility_name": "Winton Solar Farm"},
    {"state": "WA",  "cer_facility_name": "Warradarge Wind Farm Stage 2",  "nger_facility_name": "Warradarge Wind Farm"},
]


def load_overrides(base_dir):
    """
    Load manually curated CER->NGER mappings from embedded defaults plus CSV files.
    Accepted file locations (if they exist):
    - ./cer_match_overrides.csv
    - ./Task2_output/cer_match_overrides.csv
    - ./Task2_output/cer_match_overrides_approved.csv

    Required logical columns (names can vary):
    - state (optional)
    - CER facility name
    - NGER facility name
    """
    candidate_paths = [
        base_dir / "cer_match_overrides.csv",
        base_dir / "Task2_output" / "cer_match_overrides.csv",
        base_dir / "Task2_output" / "cer_match_overrides_approved.csv",
    ]

    loaded_frames = []

    if EMBEDDED_APPROVED_OVERRIDES:
        emb = pd.DataFrame(EMBEDDED_APPROVED_OVERRIDES).copy()
        emb["state"] = emb["state"].map(normalize_state)
        emb["cer_facility_name"] = emb["cer_facility_name"].astype(str).str.strip()
        emb["nger_facility_name"] = emb["nger_facility_name"].astype(str).str.strip()
        emb["override_source_file"] = "embedded_default"
        loaded_frames.append(emb[["state", "cer_facility_name", "nger_facility_name", "override_source_file"]])

    for p in candidate_paths:
        if not p.exists():
            continue

        df = pd.read_csv(p)
        cols = {c.lower().strip(): c for c in df.columns}

        def pick(*names):
            for n in names:
                if n in cols:
                    return cols[n]
            return None

        c_state = pick("state")
        c_cer = pick("cer_facility_name", "cer_name", "facility_name_cer")
        c_nger = pick("nger_facility_name", "nger_name", "facility_name_nger")

        if c_cer is None or c_nger is None:
            print(f"[warn] Skip override file with missing columns: {p}")
            continue

        out = pd.DataFrame()
        out["state"] = df[c_state].map(normalize_state) if c_state else pd.NA
        out["cer_facility_name"] = df[c_cer].astype(str).str.strip()
        out["nger_facility_name"] = df[c_nger].astype(str).str.strip()
        out["override_source_file"] = str(p)
        loaded_frames.append(out)

    if not loaded_frames:
        return pd.DataFrame(columns=["state", "cer_facility_name", "nger_facility_name", "override_source_file"])

    merged = pd.concat(loaded_frames, ignore_index=True)
    merged = merged.dropna(subset=["cer_facility_name", "nger_facility_name"]).drop_duplicates([
        "state", "cer_facility_name", "nger_facility_name"
    ])
    return merged

overrides = load_overrides(BASE_DIR)
print("Manual overrides loaded:", len(overrides))
if len(overrides):
    print(overrides.head(10).to_string(index=False))


## 6) Constrained Multi-Pass Matcher

What do in this section:
- run exact matching first.
- Then run split and fuzzy passes with guard rules.
- only compare rows in same state and compatible fuel type.
- check lexical anchors and number compatibility.
- keep review candidates for uncertain matches.

also add time interpretation after matching:
- `temporal_relation` tells if CER timing overlaps NGER history.
- `prospective_match` marks future-facing links (for example approved/probable).



In [ ]:
# Main multi-pass matcher: strict pass first, then relaxed pass.
# Goal: improve recall while still controlling wrong matches.
STRICT_THRESH = 80
RESCUE_THRESH = 76
RESCUE_MARGIN = 12
HIGH_CONF_SCORE = 90

# Future-status projects (approved/probable) use stricter fuzzy gates.
FUTURE_STRICT_THRESH = 90
FUTURE_STRICT_MARGIN = 10
FUTURE_STRICT_OVERLAP = 2
FUTURE_RESCUE_THRESH = 92
FUTURE_RESCUE_MARGIN = 12
FUTURE_RESCUE_OVERLAP = 2

REVIEW_LOW = 75
REVIEW_HIGH = 89
TOPK_CANDIDATES = 3

nger_lookup = nger_fac.copy()
nger_lookup["nger_key"] = nger_lookup["state"].astype(str) + "||" + nger_lookup["facility_name"].astype(str)

cer_lookup_all = cer_fac.copy()
cer_lookup = cer_fac[cer_fac["is_matchable_to_nger"] == True].copy()
cer_lookup["cer_key"] = cer_lookup["state"].astype(str) + "||" + cer_lookup["facility_name"].astype(str)

exact_index = {}
parts_index = {}
for _, r in nger_lookup.iterrows():
    st = r["state"]
    fc = r["fuel_category"]
    key_main = (st, fc, r["facility_name_norm_strict"])
    exact_index.setdefault(key_main, []).append(r)
    for p in r["name_parts"]:
        key_part = (st, fc, p)
        parts_index.setdefault(key_part, []).append(r)

override_map = {}
for _, r in overrides.iterrows():
    st = r["state"]
    cer_n = str(r["cer_facility_name"])
    nger_n = str(r["nger_facility_name"])
    override_map[(st, cer_n)] = nger_n

match_rows = []
review_rows = []
candidate_rows = []

def pick_topk(scored, mode, cer_row, k=TOPK_CANDIDATES):
    out = []
    for rank, (score, n, overlap) in enumerate(scored[:k], start=1):
        out.append({
            "cer_key": cer_row["cer_key"],
            "cer_state": cer_row["state"],
            "cer_facility_name": cer_row["facility_name"],
            "candidate_rank": rank,
            "candidate_mode": mode,
            "candidate_score": float(score),
            "candidate_overlap": int(overlap),
            "candidate_nger_state": n["state"],
            "candidate_nger_facility_name": n["facility_name"],
            "candidate_nger_fuel_category": n["fuel_category"],
        })
    return out

for _, c in cer_lookup.iterrows():
    c_state = c["state"]
    c_name = c["facility_name"]
    c_fc = c["fuel_category"]
    c_strict = c["facility_name_norm_strict"]
    c_relaxed = c["facility_name_norm_relaxed"]

    manual_target = override_map.get((c_state, c_name))
    if manual_target is not None:
        cand = nger_lookup[(nger_lookup["facility_name"] == manual_target) & (nger_lookup["state"] == c_state)]
        if len(cand) > 0:
            r = cand.iloc[0]
            match_rows.append({
                "cer_key": c["cer_key"], "cer_state": c_state, "cer_facility_name": c_name,
                "nger_state": r["state"], "nger_facility_name": r["facility_name"],
                "match_score": 100.0, "match_margin": 100.0, "match_overlap": 999,
                "match_pass": "manual_override", "match_method": "override", "confidence_tier": "high"
            })
            continue

    pool = nger_lookup[nger_lookup["state"] == c_state].copy()
    if c_fc is not None:
        constrained = pool[pool["fuel_category"] == c_fc]
        if len(constrained) > 0:
            pool = constrained

    if len(pool) == 0:
        review_rows.append({
            "cer_state": c_state, "cer_facility_name": c_name, "reason": "no_candidates",
            "best_score": np.nan, "second_score": np.nan, "margin": np.nan, "best_overlap": np.nan,
            "suggested_nger": pd.NA
        })
        continue

    p1 = exact_index.get((c_state, c_fc, c_strict), [])
    if not p1 and c_fc is not None:
        p1 = exact_index.get((c_state, None, c_strict), [])
    if p1:
        r = p1[0]
        if numeric_compatible(c_name, r["facility_name"]) and lexical_guard(c_strict, r["facility_name_norm_strict"], mode="strict"):
            match_rows.append({
                "cer_key": c["cer_key"], "cer_state": c_state, "cer_facility_name": c_name,
                "nger_state": r["state"], "nger_facility_name": r["facility_name"],
                "match_score": 100.0, "match_margin": 100.0, "match_overlap": 999,
                "match_pass": "pass1_exact", "match_method": "strict_exact", "confidence_tier": "high"
            })
            continue

    found = None
    for part in c["name_parts"]:
        p2 = parts_index.get((c_state, c_fc, part), [])
        if not p2 and c_fc is not None:
            p2 = parts_index.get((c_state, None, part), [])
        if p2:
            for candidate in p2:
                if numeric_compatible(c_name, candidate["facility_name"]) and lexical_guard(part, candidate["facility_name_norm_strict"], mode="strict"):
                    found = candidate
                    break
        if found is not None:
            break

    if found is not None:
        match_rows.append({
            "cer_key": c["cer_key"], "cer_state": c_state, "cer_facility_name": c_name,
            "nger_state": found["state"], "nger_facility_name": found["facility_name"],
            "match_score": 95.0, "match_margin": 95.0, "match_overlap": 999,
            "match_pass": "pass2_split_exact", "match_method": "split_exact", "confidence_tier": "high"
        })
        continue

    cer_is_future = str(c.get("project_status", "")).strip().lower() in {"approved", "probable"}

    scored = []
    for _, n in pool.iterrows():
        if not numeric_compatible(c_name, n["facility_name"]):
            continue
        if not lexical_guard(c_strict, n["facility_name_norm_strict"], mode="strict"):
            continue
        overlap = lexical_overlap_count(c_strict, n["facility_name_norm_strict"])
        s = fuzz.token_sort_ratio(c_strict, n["facility_name_norm_strict"])
        scored.append((s, n, overlap))

    scored.sort(key=lambda x: x[0], reverse=True)
    candidate_rows.extend(pick_topk(scored, "strict", c, TOPK_CANDIDATES))

    top1 = scored[0][0] if scored else -1
    top2 = scored[1][0] if len(scored) > 1 else -1
    top1_overlap = scored[0][2] if scored else 0
    margin = top1 - top2 if top2 >= 0 else top1

    strict_accept = False
    if scored:
        if cer_is_future:
            strict_accept = (
                top1 >= FUTURE_STRICT_THRESH and
                top1_overlap >= FUTURE_STRICT_OVERLAP and
                margin >= FUTURE_STRICT_MARGIN
            )
        else:
            strict_accept = (top1 >= STRICT_THRESH and top1_overlap >= 1 and margin >= 8)

    if strict_accept:
        best = scored[0][1]
        if cer_is_future:
            tier = "high" if top1 >= 95 else "medium"
            pass_name = "pass3_strict_fuzzy_future_guarded"
            method_name = "token_sort_ratio_future_guarded"
        else:
            tier = "high" if top1 >= HIGH_CONF_SCORE else "medium"
            pass_name = "pass3_strict_fuzzy"
            method_name = "token_sort_ratio"

        match_rows.append({
            "cer_key": c["cer_key"], "cer_state": c_state, "cer_facility_name": c_name,
            "nger_state": best["state"], "nger_facility_name": best["facility_name"],
            "match_score": float(top1), "match_margin": float(margin), "match_overlap": int(top1_overlap),
            "match_pass": pass_name, "match_method": method_name, "confidence_tier": tier
        })
        continue

    scored_relaxed = []
    for _, n in pool.iterrows():
        if not numeric_compatible(c_name, n["facility_name"]):
            continue
        if not lexical_guard(c_relaxed, n["facility_name_norm_relaxed"], mode="rescue"):
            continue

        overlap = lexical_overlap_count(c_relaxed, n["facility_name_norm_relaxed"])
        s = fuzz.token_set_ratio(c_relaxed, n["facility_name_norm_relaxed"])

        if overlap == 1 and s < 92:
            continue
        scored_relaxed.append((s, n, overlap))

    scored_relaxed.sort(key=lambda x: x[0], reverse=True)
    candidate_rows.extend(pick_topk(scored_relaxed, "rescue", c, TOPK_CANDIDATES))

    r1 = scored_relaxed[0][0] if scored_relaxed else -1
    r2 = scored_relaxed[1][0] if len(scored_relaxed) > 1 else -1
    r1_overlap = scored_relaxed[0][2] if scored_relaxed else 0
    rmargin = r1 - r2 if r2 >= 0 else r1

    rescue_accept = False
    if scored_relaxed:
        if cer_is_future:
            rescue_accept = (
                r1 >= FUTURE_RESCUE_THRESH and
                rmargin >= FUTURE_RESCUE_MARGIN and
                r1_overlap >= FUTURE_RESCUE_OVERLAP
            )
        else:
            if r1 >= RESCUE_THRESH and rmargin >= RESCUE_MARGIN and r1_overlap >= 2:
                rescue_accept = True
            elif r1 >= 88 and r1_overlap >= 1 and rmargin >= 8:
                rescue_accept = True

    if rescue_accept:
        best = scored_relaxed[0][1]
        if cer_is_future:
            tier = "high" if r1 >= 95 else "medium"
            pass_name = "pass4_rescue_future_guarded"
            method_name = "token_set_ratio_relaxed_future_guarded"
        else:
            tier = "high" if r1 >= HIGH_CONF_SCORE else "medium"
            pass_name = "pass4_rescue"
            method_name = "token_set_ratio_relaxed"

        match_rows.append({
            "cer_key": c["cer_key"], "cer_state": c_state, "cer_facility_name": c_name,
            "nger_state": best["state"], "nger_facility_name": best["facility_name"],
            "match_score": float(r1), "match_margin": float(rmargin), "match_overlap": int(r1_overlap),
            "match_pass": pass_name, "match_method": method_name, "confidence_tier": tier
        })
    else:
        best_name = scored_relaxed[0][1]["facility_name"] if scored_relaxed else (scored[0][1]["facility_name"] if scored else pd.NA)
        best_score = r1 if scored_relaxed else (top1 if scored else np.nan)
        second_score = r2 if scored_relaxed else (top2 if len(scored) > 1 else np.nan)
        best_overlap = r1_overlap if scored_relaxed else top1_overlap
        best_margin = rmargin if scored_relaxed else margin
        review_reason = "future_no_confident_match" if cer_is_future else "no_confident_match"
        review_rows.append({
            "cer_state": c_state,
            "cer_facility_name": c_name,
            "reason": review_reason,
            "best_score": best_score,
            "second_score": second_score,
            "margin": best_margin,
            "best_overlap": best_overlap,
            "suggested_nger": best_name,
        })

match_map = pd.DataFrame(match_rows)


if len(match_map):
    cer_meta = (
        cer[["state", "facility_name", "project_status", "accreditation_year"]]
        .dropna(subset=["state", "facility_name"])
        .drop_duplicates(["state", "facility_name"])
        .rename(columns={
            "state": "cer_state",
            "facility_name": "cer_facility_name",
            "project_status": "cer_project_status",
            "accreditation_year": "cer_accreditation_year",
        })
    )
    match_map = match_map.merge(cer_meta, on=["cer_state", "cer_facility_name"], how="left")

    nger_span = (
        nger[["state", "facility_name", "year"]]
        .dropna(subset=["state", "facility_name", "year"])
        .groupby(["state", "facility_name"], as_index=False)
        .agg(nger_year_min=("year", "min"), nger_year_max=("year", "max"))
        .rename(columns={"state": "nger_state", "facility_name": "nger_facility_name"})
    )
    match_map = match_map.merge(nger_span, on=["nger_state", "nger_facility_name"], how="left")

    cer_y = pd.to_numeric(match_map["cer_accreditation_year"], errors="coerce")
    ymin = pd.to_numeric(match_map["nger_year_min"], errors="coerce")
    ymax = pd.to_numeric(match_map["nger_year_max"], errors="coerce")

    c_unknown = (cer_y.isna() | ymin.isna() | ymax.isna()).to_numpy()
    c_overlap = ((cer_y >= ymin) & (cer_y <= ymax)).fillna(False).to_numpy()
    c_after = (cer_y > ymax).fillna(False).to_numpy()
    c_before = (cer_y < ymin).fillna(False).to_numpy()

    match_map["temporal_relation"] = np.select(
        [c_unknown, c_overlap, c_after, c_before],
        ["unknown", "overlap", "cer_after_nger", "cer_before_nger"],
        default="unknown",
    )

    status_lc = match_map["cer_project_status"].astype("string").str.lower()
    match_map["prospective_match"] = status_lc.isin(["approved", "probable"]) | match_map["temporal_relation"].eq("cer_after_nger")

review_queue = pd.DataFrame(review_rows)
candidate_topk = pd.DataFrame(candidate_rows)

low_score_review = review_queue[review_queue["best_score"].between(REVIEW_LOW, REVIEW_HIGH, inclusive="both")].copy()

print("Matched CER facilities (eligible):", len(match_map), "/", len(cer_lookup))
print("CER facilities excluded from matching (non-comparable):", len(cer_lookup_all) - len(cer_lookup))
if len(match_map):
    print(match_map["match_pass"].value_counts())
    print("Confidence tiers:")
    print(match_map["confidence_tier"].value_counts())
print("Review queue size:", len(review_queue), "low-score({}-{}):".format(REVIEW_LOW, REVIEW_HIGH), len(low_score_review))


## 7) Join Back to Row-Level and Integrate ABS

What do in this section:
- map facility-level match results back to full NGER/CER rows.
- merge ABS attributes by `state` + `year`.
- produce one denormalized table for downstream analysis.

Why this matters:
- need one query-ready table for Stage 4 SQL and reporting.
- It also makes QA easier because all key columns are together.


In [ ]:
# Join facility-level matches back to row-level data.
# Then combine with ABS to build the integrated table.
cer_for_join = cer.merge(
    match_map,
    how="left",
    left_on=["state", "facility_name"],
    right_on=["cer_state", "cer_facility_name"],
)

nger_norm_index = nger[["state", "facility_name", "facility_name_norm_strict"]].drop_duplicates()
cer_for_join = cer_for_join.merge(
    nger_norm_index.rename(columns={"facility_name": "nger_facility_name", "facility_name_norm_strict": "matched_nger_norm"}),
    how="left", on=["state", "nger_facility_name"]
)

cer_for_join["join_facility_norm"] = cer_for_join["matched_nger_norm"].combine_first(cer_for_join["facility_name_norm_strict"])
nger_for_join = nger.copy()
nger_for_join["join_facility_norm"] = nger_for_join["facility_name_norm_strict"]

merged = nger_for_join.add_prefix("nger_").merge(
    cer_for_join.add_prefix("cer_"),
    left_on=["nger_state", "nger_join_facility_norm"],
    right_on=["cer_state", "cer_join_facility_norm"],
    how="outer",
)

merged["facility_name"] = merged["nger_facility_name"].combine_first(merged["cer_facility_name"])
merged["state"] = merged["nger_state"].combine_first(merged["cer_state"])
merged["year"] = merged["nger_year"]

has_nger = merged["nger_reporting_year"].notna() | merged["nger_entity_name"].notna() | merged["nger_facility_name"].notna()
has_cer = merged["cer_project_status"].notna() | merged["cer_facility_name"].notna()
merged["data_source"] = np.select(
    [has_nger & has_cer, has_nger, has_cer],
    ["NGER_CER_matched", "NGER_only", "CER_only"],
    default="unknown",
)

final = merged.merge(abs_clean.add_prefix("abs_"), left_on=["state", "year"], right_on=["abs_state", "abs_year"], how="left")

print("Final shape:", final.shape)
print(final["data_source"].value_counts(dropna=False))
print("Rows with CER fuzzy/override score:", int(final["cer_match_score"].notna().sum()))

## 8) Validation (Coverage + Confidence + Review Assets)

What do in this section:
- report matching coverage for full and eligible subsets.
- inspect match methods, score tiers, and unresolved cases.
- check status mix (`approved` / `committed` / `probable`) in matched rows.
- check temporal labels (`overlap`, `cer_after_nger`, etc.).
- export review templates for manual checking.



In [ ]:
# Validation: coverage, confidence, and review assets.
# These checks help us inspect match quality.
total_cer_overall = len(cer_lookup_all)
total_cer_eligible = len(cer_lookup)
matched_cer = len(match_map)
coverage_overall = (matched_cer / total_cer_overall) if total_cer_overall else np.nan
coverage_eligible = (matched_cer / total_cer_eligible) if total_cer_eligible else np.nan

validation_summary = pd.DataFrame([
    {"metric": "cer_unique_facilities_overall", "value": total_cer_overall},
    {"metric": "cer_unique_facilities_eligible", "value": total_cer_eligible},
    {"metric": "matched_unique_facilities", "value": matched_cer},
    {"metric": "match_coverage_overall", "value": coverage_overall},
    {"metric": "match_coverage_eligible", "value": coverage_eligible},
    {"metric": "review_queue_size", "value": len(review_queue)},
    {"metric": "candidate_topk_rows", "value": len(candidate_topk)},
])

print("Validation summary:")
print(validation_summary)

if len(match_map):
    pass_dist = match_map.groupby(["match_pass", "confidence_tier"]).size().reset_index(name="count")
    print("\nPass x tier distribution:")
    print(pass_dist.sort_values(["match_pass", "confidence_tier"]).to_string(index=False))

medium_sample = match_map[match_map["confidence_tier"] == "medium"].copy() if len(match_map) else pd.DataFrame()
medium_sample = medium_sample.head(80)

borderline_review = review_queue[review_queue["best_score"].between(70, 89, inclusive="both")].copy() if len(review_queue) else pd.DataFrame()
borderline_review = borderline_review.head(120)

manual_validation_template = pd.DataFrame()
if len(medium_sample):
    m = medium_sample[["cer_state", "cer_facility_name", "nger_facility_name", "match_score", "match_pass", "match_method", "confidence_tier"]].copy()
    m["validation_source"] = "auto_matched_medium"
    manual_validation_template = pd.concat([manual_validation_template, m], ignore_index=True)
if len(borderline_review):
    r = borderline_review[["cer_state", "cer_facility_name", "suggested_nger", "best_score", "reason"]].copy()
    r = r.rename(columns={"suggested_nger": "nger_facility_name", "best_score": "match_score", "reason": "match_pass"})
    r["match_method"] = "candidate_from_review"
    r["confidence_tier"] = "review"
    r["validation_source"] = "unmatched_borderline"
    manual_validation_template = pd.concat([manual_validation_template, r], ignore_index=True)

if len(manual_validation_template):
    manual_validation_template["human_label"] = ""  # correct / wrong / unsure
    manual_validation_template["human_correct_nger_name"] = ""
    manual_validation_template["human_notes"] = ""

print("\nManual validation template rows:", len(manual_validation_template))

# Unmatched reason distribution (for report)
unmatched_reason_dist = pd.DataFrame()
if len(review_queue):
    unmatched_reason_dist = (
        review_queue["reason"]
        .fillna("unknown")
        .value_counts(dropna=False)
        .rename_axis("reason")
        .reset_index(name="count")
    )

print("\nUnmatched reason distribution:")
if len(unmatched_reason_dist):
    print(unmatched_reason_dist.to_string(index=False))
else:
    print("No unmatched review rows.")

# Coverage by CER project status (approved/committed/probable)
status_coverage = pd.DataFrame()
if len(cer_lookup_all):
    status_base = cer_lookup_all[["state", "facility_name", "project_status"]].drop_duplicates().copy()
    status_base["cer_key"] = status_base["state"].astype(str) + "||" + status_base["facility_name"].astype(str)

    matched_key_set = set(match_map["cer_key"]) if len(match_map) else set()
    status_base["is_matched"] = status_base["cer_key"].isin(matched_key_set)

    status_coverage = (
        status_base
        .groupby("project_status", dropna=False)
        .agg(total_cer=("cer_key", "nunique"), matched_cer=("is_matched", "sum"))
        .reset_index()
    )
    status_coverage["coverage"] = status_coverage["matched_cer"] / status_coverage["total_cer"]
    status_coverage = status_coverage.sort_values(["project_status"], na_position="last")

print("\nCoverage by CER project status:")
if len(status_coverage):
    print(status_coverage.to_string(index=False))
else:
    print("No CER status coverage available.")

# Save only the two small validation tables needed in report
if len(unmatched_reason_dist):
    unmatched_reason_dist.to_csv(TASK2_DIR / "validation_unmatched_reason_distribution.csv", index=False)
if len(status_coverage):
    status_coverage.to_csv(TASK2_DIR / "validation_coverage_by_status.csv", index=False)

gold_path = BASE_DIR / "Task2_output" / "cer_gold_labels.csv"
if gold_path.exists():
    gold = pd.read_csv(gold_path)
    pred = match_map[["cer_state", "cer_facility_name", "nger_facility_name"]].drop_duplicates()
    merged_gold = gold.merge(pred, on=["cer_state", "cer_facility_name"], how="left")
    merged_gold["is_correct"] = (merged_gold["gold_nger_facility_name"].astype(str).str.strip().str.lower() ==
                                 merged_gold["nger_facility_name"].astype(str).str.strip().str.lower())
    precision_sample = merged_gold["is_correct"].mean() if len(merged_gold) else np.nan
    print("\nGold-label precision on provided sample:", precision_sample)
else:
    print("\nNo gold label file found at Task2_output/cer_gold_labels.csv (optional).")


## Generate Override Draft

Create a high-confidence draft override file for manual review. Approve rows by filling `human_decision`.

In [ ]:
# Create a high-confidence override draft for review.
# This is a suggestion list, not final auto-approval.
if len(candidate_topk):
    top1 = (
        candidate_topk
        .sort_values(["cer_key", "candidate_score", "candidate_overlap"], ascending=[True, False, False])
        .drop_duplicates(["cer_key"], keep="first")
        .copy()
    )

    override_draft = top1[
        (top1["candidate_score"] >= 80) &
        (top1["candidate_overlap"] >= 1)
    ].copy()

    override_draft = override_draft.rename(columns={
        "cer_state": "state",
        "candidate_nger_facility_name": "nger_facility_name",
    })

    keep_cols = [
        "state", "cer_facility_name", "nger_facility_name",
        "candidate_score", "candidate_overlap", "candidate_mode"
    ]
    override_draft = override_draft[[c for c in keep_cols if c in override_draft.columns]]
    override_draft["human_decision"] = ""   # approve / reject
    override_draft["human_notes"] = ""
else:
    override_draft = pd.DataFrame(columns=[
        "state", "cer_facility_name", "nger_facility_name",
        "candidate_score", "candidate_overlap", "candidate_mode",
        "human_decision", "human_notes"
    ])

override_draft_path = TASK2_DIR / "cer_match_overrides_draft.csv"
override_approved_path = TASK2_DIR / "cer_match_overrides_approved.csv"
override_draft.to_csv(override_draft_path, index=False)
print("Override draft rows:", len(override_draft))
print("Draft file:", override_draft_path)
print("To apply, copy approved rows to:", override_approved_path)
override_draft.head(20)


## 9) Suggest Overrides for Human Review

What this section does:
- Builds a **suggested** override list from top candidate matches that are strong but not auto-accepted.


In [ ]:
# Suggest overrides for currently unmatched CER facilities.
# Use conservative filters to reduce false positives.
matched_keys = set(match_map["cer_key"]) if len(match_map) else set()

if len(candidate_topk):
    top1 = (
        candidate_topk
        .sort_values(["cer_key", "candidate_score", "candidate_overlap"], ascending=[True, False, False])
        .drop_duplicates(["cer_key"], keep="first")
        .copy()
    )

    suggested_overrides = top1[
        (~top1["cer_key"].isin(matched_keys)) &
        (top1["candidate_score"] >= 85) &
        (top1["candidate_overlap"] >= 2)
    ].copy()

    suggested_overrides = suggested_overrides.rename(columns={
        "cer_state": "state",
        "cer_facility_name": "cer_facility_name",
        "candidate_nger_facility_name": "nger_facility_name",
        "candidate_score": "suggested_score",
        "candidate_mode": "suggested_mode",
    })

    keep_cols = [
        "state", "cer_facility_name", "nger_facility_name",
        "suggested_score", "suggested_mode", "candidate_overlap"
    ]
    suggested_overrides = suggested_overrides[[c for c in keep_cols if c in suggested_overrides.columns]].copy()
    suggested_overrides["human_decision"] = ""   # approve / reject
    suggested_overrides["human_notes"] = ""

else:
    suggested_overrides = pd.DataFrame(columns=[
        "state", "cer_facility_name", "nger_facility_name",
        "suggested_score", "suggested_mode", "candidate_overlap",
        "human_decision", "human_notes"
    ])

print("Suggested override rows:", len(suggested_overrides))
if len(suggested_overrides):
    print(suggested_overrides.head(20).to_string(index=False))


## 10) Save Outputs

This section writes all final deliverables and audit assets to disk.

In [ ]:
# Save all Task2 output files.
# If a file is locked, write a *_latest fallback file.
def safe_write_csv(df, path_obj):
    path_obj = Path(path_obj)
    try:
        df.to_csv(path_obj, index=False)
        return str(path_obj)
    except PermissionError:
        fallback = path_obj.with_name(path_obj.stem + "_latest" + path_obj.suffix)
        df.to_csv(fallback, index=False)
        print(f"[warn] File locked, wrote fallback: {fallback}")
        return str(fallback)

written = []

written.append(safe_write_csv(nger, TASK2_DIR / "clean_nger_facility_year.csv"))
written.append(safe_write_csv(cer, TASK2_DIR / "clean_cer_power_stations.csv"))
written.append(safe_write_csv(abs_clean, TASK2_DIR / "clean_abs_state_year_population_people.csv"))

final_clean = final.copy()

logical_drop_cols = [
    "nger_facility_name", "cer_cer_facility_name", "cer_nger_facility_name",
    "nger_state", "cer_state", "cer_cer_state", "cer_nger_state", "abs_state",
    "nger_year", "abs_year", "abs_abs_people_Year",
]
for c in logical_drop_cols:
    if c in final_clean.columns:
        final_clean = final_clean.drop(columns=[c])

INTERMEDIATE_BLACKLIST = {
    "nger_facility_name_norm_strict", "nger_facility_name_norm_relaxed",
    "cer_facility_name_norm_strict", "cer_facility_name_norm_relaxed",
    "nger_name_parts", "nger_name_numbers", "cer_name_parts", "cer_name_numbers",
    "nger_join_facility_norm", "cer_join_facility_norm",
    "cer_match_margin", "cer_match_overlap", "cer_match_pass", "cer_match_method",
    "cer_confidence_tier", "cer_matched_nger_norm",
    "match_margin", "match_overlap", "match_pass", "match_method", "confidence_tier", "matched_nger_norm",
    "cer_cer_key", "cer_key", "nger_key",
}

existing_drop = [c for c in final_clean.columns if c in INTERMEDIATE_BLACKLIST]
if existing_drop:
    final_clean = final_clean.drop(columns=existing_drop)

HELPER_PATTERNS = [
    "_norm_", "_join_", "name_parts", "name_numbers",
    "match_margin", "match_overlap", "match_pass", "match_method", "confidence_tier", "matched_nger_norm"
]
pattern_drop = []
for c in final_clean.columns:
    lc = c.lower()
    if any(p in lc for p in HELPER_PATTERNS):
        pattern_drop.append(c)
if pattern_drop:
    final_clean = final_clean.drop(columns=sorted(set(pattern_drop)))

rename_map = {}
for c in final_clean.columns:
    nc = c
    if nc.startswith("abs_abs_people_"):
        nc = "abs_people_" + nc[len("abs_abs_people_"):]
    if nc.startswith("abs_abs_"):
        nc = "abs_" + nc[len("abs_abs_"):]
    if nc.startswith("cer_cer_"):
        nc = "cer_" + nc[len("cer_cer_"):]
    rename_map[c] = nc
final_clean = final_clean.rename(columns=rename_map)

final_clean = final_clean.loc[:, ~final_clean.columns.duplicated()]

front_cols = [
    "facility_name", "cer_facility_name", "state", "year", "data_source",
    "nger_reporting_year", "cer_project_status",
    "cer_accreditation_start_date", "cer_approval_date", "cer_committed_date", "cer_reference_date", "cer_accreditation_year"
]
front_cols = [c for c in front_cols if c in final_clean.columns]
other_cols = [c for c in final_clean.columns if c not in front_cols]
final_clean = final_clean[front_cols + other_cols]

written.append(safe_write_csv(final_clean, TASK2_DIR / "fact_annual_clean.csv"))

print("Saved minimal outputs to:", TASK2_DIR)
print("fact_annual_clean shape:", final_clean.shape)
print("written files:")
for w in written:
    print(" -", w)


# Stage 3 · Add coordinates

With a configured Nominatim-compatible search service, try the facility name,
postcode and state first, then fall back to postcode and state. A fallback is
an approximate location. When no service is configured, leave coordinates null
and continue with the DuckDB load.


In [ ]:
import pandas as pd
import requests
import time as t
from functools import lru_cache

# Setup Directories
TASK2_DIR = BATCH_DIR / "Task2_output"
TASK3_DIR = BATCH_DIR / "Task3_output"
TASK3_DIR.mkdir(parents=True, exist_ok=True)

# Load the cleaned CER power stations table from Stage 2
cer_df_to_geocode = pd.read_csv(TASK2_DIR / "clean_cer_power_stations.csv")

GEOCODER_URL = os.getenv("NOMINATIM_SEARCH_URL")

@lru_cache(maxsize=None)
def address_details(query):
    if not GEOCODER_URL:
        return []
    t.sleep(2)
    response = requests.get(
        GEOCODER_URL,
        params={"q": query, "format": "json", "addressdetails": 0},
        headers={"User-Agent": "energy-data-pipeline/1.0"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

# Geocoding Loop
latitudes = []
longitudes = []

if GEOCODER_URL:
    print(f"Starting geocoding for {len(cer_df_to_geocode)} power stations.")
else:
    print("No geocoding service configured. Coordinates will be null.")

for index, row in cer_df_to_geocode.iterrows():
    name = row['facility_name']
    postcode = row['postcode']
    state = row['state']
    
    # Attempt 1: Search using facility name, postcode, and state
    results = address_details(f"{name}, {postcode}, {state}, Australia")
    
    if not results and pd.notna(postcode) and pd.notna(state):
        results = address_details(f"{postcode}, {state}, Australia")
    
    # Extract coordinates if any result was found
    if len(results) > 0:
        latitudes.append(results[0]['lat'])
        longitudes.append(results[0]['lon'])
    else:
        latitudes.append(None)
        longitudes.append(None)
            
    if (index + 1) % 20 == 0:
        print(f"Processed {index + 1}/{len(cer_df_to_geocode)} facilities...")

# Store Results and Print Summary (New and Improved Version)
cer_augmented_df = cer_df_to_geocode.copy()
# Important: convert to numeric, coercing errors to NaN for proper counting
cer_augmented_df['latitude'] = pd.to_numeric(latitudes, errors='coerce')
cer_augmented_df['longitude'] = pd.to_numeric(longitudes, errors='coerce')


augmented_path = TASK3_DIR / "augmented_cer_power_stations_with_coords.csv"
cer_augmented_df.to_csv(augmented_path, index=False)

print(f"\nGeocoding completed. Augmented dataset saved to: {augmented_path}")

# Coordinate coverage
print("\n\n=== GEOCODING RESULTS ===")
total = len(cer_augmented_df)
got = int(cer_augmented_df['latitude'].notna().sum())
null = int(cer_augmented_df['latitude'].isna().sum())
print(f"Total stations: {total}")
print(f"Got coordinates: {got} ({got/total*100:.1f}%)")
print(f"Null (no match): {null} ({null/total*100:.1f}%)")
print("-" * 35)
print("\nSample (first 5 with coords):")
print(cer_augmented_df[cer_augmented_df['latitude'].notna()]
      [['facility_name','state','latitude','longitude']].head(5).to_string(index=False))
print("\n" + "-" * 35)
print("\nNull coordinate stations:")
# Check if there are any nulls before printing
if null > 0:
    print(cer_augmented_df[cer_augmented_df['latitude'].isna()]
          [['facility_name','state']].to_string(index=False))
else:
    print("All stations were successfully geocoded!")
print("=" * 35)


In [ ]:
import folium

# Use the geocoded dataframe from the previous step
# Ensure the dataframe is named cer_augmented_df from the previous cell
map_df = cer_augmented_df.dropna(subset=['latitude', 'longitude']).copy()

# Define a color mapping for different fuel categories
def get_color(fuel_category):
    if 'Solar' in str(fuel_category):
        return 'orange'
    elif 'Wind' in str(fuel_category):
        return 'blue'
    elif 'Hydro' in str(fuel_category):
        return 'lightblue'
    elif 'Biomass' in str(fuel_category) or 'Bioenergy' in str(fuel_category):
        return 'green'
    else:
        return 'gray' # Default color for other types

map_df['color'] = map_df['fuel_category'].apply(get_color)

# Create a base map centered on Australia
aus_map = folium.Map(location=[-25.27, 133.77], zoom_start=4)

# Add a point for each power station
for index, row in map_df.iterrows():
    # Create a popup with information
    popup_html = f"""
    <b>Name:</b> {row['facility_name']}<br>
    <b>Fuel:</b> {row['fuel_category']}<br>
    <b>Status:</b> {row['project_status']}
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=row['color'],
        fill=True,
        fill_color=row['color'],
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(aus_map)

# Display the map
print("Interactive map of geocoded CER power stations by fuel type:")
aus_map


## Stage 4: Load Big Wide Table + Coordinates into DuckDB

In this section, Stage 4 builds a **single denormalized wide table** by combining:
- Stage 2 big table: `fact_annual_clean.csv`
- Stage 3 location table: `augmented_cer_power_stations.csv` (only `latitude` and `longitude`)

What this section does:
- Creates a persistent DuckDB database file.
- Loads the Stage 2 big table and Stage 3 location table.
- Left-joins only `latitude` and `longitude` by `cer_unique_key`.
- Creates `integrated_energy_wide` in DuckDB from the merged dataframe.
- Builds optional geometry column `geom` if DuckDB spatial extension is available.


In [ ]:
from pathlib import Path
import pandas as pd
import duckdb

# Paths
BASE_DIR = BATCH_DIR
TASK2_DIR = BASE_DIR / "Task2_output"
TASK4_DIR = BASE_DIR / "Task4_output"
TASK3_DIR = BASE_DIR / "Task3_output"
DB_PATH = TASK4_DIR / "energy_task4.duckdb"
WIDE_CSV = TASK2_DIR / "fact_annual_clean.csv"
LOCATION_CSV = TASK3_DIR / "augmented_cer_power_stations_with_coords.csv"

TASK4_DIR.mkdir(parents=True, exist_ok=True)

print("DB_PATH:", DB_PATH)
print("WIDE_CSV exists:", WIDE_CSV.exists())
print("LOCATION_CSV exists:", LOCATION_CSV.exists())

# Load Stage 2 big wide table and Stage 3 location table
wide_df = pd.read_csv(WIDE_CSV)
loc_df = pd.read_csv(LOCATION_CSV)

# Keep one location record per cer_unique_key
loc_df = loc_df.dropna(subset=["cer_unique_key"]).drop_duplicates(subset=["cer_unique_key"])

# Only keep required coordinate fields from Stage 3
location_cols = ["cer_unique_key", "latitude", "longitude"]
location_cols = [c for c in location_cols if c in loc_df.columns]
loc_merge_df = loc_df[location_cols].copy()

# Merge coordinates into the big table
wide_df = wide_df.merge(loc_merge_df, on="cer_unique_key", how="left", validate="many_to_one")

# Force numeric coordinates for spatial conversion safety
if "latitude" not in wide_df.columns:
    wide_df["latitude"] = pd.NA
if "longitude" not in wide_df.columns:
    wide_df["longitude"] = pd.NA
wide_df["latitude"] = pd.to_numeric(wide_df["latitude"], errors="coerce")
wide_df["longitude"] = pd.to_numeric(wide_df["longitude"], errors="coerce")

print("Merged wide table shape:", wide_df.shape)
print("Rows with coordinates after merge:", int((wide_df["latitude"].notna() & wide_df["longitude"].notna()).sum()))



### Create Table in DuckDB (with Optional Spatial Geometry)

Implementation notes:
- `integrated_energy_wide` is rebuilt each run for reproducibility.
- Only `latitude` and `longitude` are merged from Stage 3 (`augmented_cer_power_stations.csv`) via `cer_unique_key`.
- `geom` is created only when DuckDB spatial extension can be loaded; otherwise table creation still succeeds.


In [ ]:
con = duckdb.connect(DB_PATH.as_posix())

# Try to enable spatial extension, but keep workflow robust if unavailable
spatial_loaded = False
try:
    con.execute("INSTALL spatial;")
    con.execute("LOAD spatial;")
    spatial_loaded = True
except Exception as e:
    print(f"Spatial extension unavailable; continuing without geom. Reason: {e}")

# Register dataframe as temporary source
con.register("wide_df", wide_df)

# Rebuild wide table for deterministic reruns
con.execute("DROP TABLE IF EXISTS integrated_energy_wide")
if spatial_loaded:
    sql = '''
    CREATE TABLE integrated_energy_wide AS
    SELECT
        w.*,
        CASE
            WHEN w.longitude IS NOT NULL AND w.latitude IS NOT NULL
            THEN ST_Point(CAST(w.longitude AS DOUBLE), CAST(w.latitude AS DOUBLE))
            ELSE NULL
        END AS geom
    FROM wide_df w
    '''
else:
    sql = '''
    CREATE TABLE integrated_energy_wide AS
    SELECT
        w.*,
        NULL AS geom
    FROM wide_df w
    '''
con.execute(sql)

# Basic validation
row_count = con.execute("SELECT COUNT(*) FROM integrated_energy_wide").fetchone()[0]
coord_count = con.execute("SELECT COUNT(*) FROM integrated_energy_wide WHERE latitude IS NOT NULL AND longitude IS NOT NULL").fetchone()[0]
geom_count = con.execute("SELECT COUNT(*) FROM integrated_energy_wide WHERE geom IS NOT NULL").fetchone()[0]

print("Table created: integrated_energy_wide")
print("Rows:", row_count)
print("Rows with coordinates:", coord_count)
print("Rows with geometry:", geom_count)
print("Spatial loaded:", spatial_loaded)

# Show key column types
print(con.execute("PRAGMA table_info('integrated_energy_wide')").df().query("name in ['latitude','longitude','geom']"))

con.close()
print("DuckDB saved at:", DB_PATH)


### Post-Load Validation Tests (DuckDB)

This test block verifies that the DuckDB file exists, the target table was created, data was loaded, and key fields are present.


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = BATCH_DIR / "Task4_output" / "energy_task4.duckdb"
TABLE_NAME = "integrated_energy_wide"
REQUIRED_FIELDS = ["cer_unique_key", "latitude", "longitude"]

assert DB_PATH.exists(), f"Database file not found: {DB_PATH}"

con = duckdb.connect(DB_PATH.as_posix())

# 1) Show all tables
tables_df = con.execute("SHOW TABLES").df()
print("Tables in database:")
print(tables_df)
assert TABLE_NAME in tables_df.iloc[:, 0].tolist(), f"Table not found: {TABLE_NAME}"

# 2) Show schema / fields
schema_df = con.execute(f"PRAGMA table_info('{TABLE_NAME}')").df()
print(f"\nFields in {TABLE_NAME}:")
print(schema_df[["name", "type"]])
field_names = schema_df["name"].tolist()

# 3) Assertions (basic tests)
for field in REQUIRED_FIELDS:
    assert field in field_names, f"Missing required field: {field}"

row_count = con.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}").fetchone()[0]
assert row_count > 0, "Table exists but contains 0 rows"
print(f"\nRow count: {row_count}")

# 4) Show sample rows
sample_df = con.execute(f"SELECT * FROM {TABLE_NAME} LIMIT 10").df()
print("\nSample rows (first 10):")
print(sample_df)

# 5) Optional quality check: non-null coordinates
coord_count = con.execute(
    f"SELECT COUNT(*) FROM {TABLE_NAME} WHERE latitude IS NOT NULL AND longitude IS NOT NULL"
).fetchone()[0]
print(f"Rows with non-null coordinates: {coord_count}")

con.close()
print("\nValidation passed: DuckDB import looks successful.")

